In [4]:
import osmnx as ox
import pandas as pd
import ast
import math 

# Toạ độ của trung tâm thành phố Hồ Chí Minh {north, south, east, west} = {10.79326, 10.74300, 106.74505, 106.67175}
osm_files = [
    'data/original/hcm_map1.osm',
    'data/original/hcm_map2.osm',
    'data/original/hcm_map3.osm',
    'data/original/hcm_map4.osm'
]
train_file = 'data/original/oritrain.csv'

drive_types = ['motorway', 'motorway_link', 'trunk', 'trunk_link', 
    'primary', 'primary_link', 'secondary', 'secondary_link', 
    'tertiary', 'tertiary_link', 'unclassified', 'residential',
    'service', 'living_street', 'pedestrian', 'track', 'road']

all_nodes = []
all_edges = []

for file in osm_files:
    G = ox.graph_from_xml(file, simplify=False)
    nodes, edges = ox.graph_to_gdfs(G)
    all_nodes.append(nodes)
    all_edges.append(edges)

nodes_combined = pd.concat(all_nodes)
edges_combined = pd.concat(all_edges)

nodes_combined = nodes_combined.reset_index().drop_duplicates(subset=['osmid'])
edges_combined = edges_combined.reset_index().drop_duplicates(subset=['u', 'v', 'key'])

edges = edges_combined[
    edges_combined['highway'].isin(drive_types) | 
    edges_combined['highway'].apply(lambda x: any(i in str(x) for i in drive_types))
].copy()

# Xử lý nodes
nodes = nodes.reset_index()
nodes_df = nodes_combined[['osmid', 'y', 'x']].copy()
nodes_df['osmid'] = nodes_df['osmid'].astype('int64')
nodes_df['y'] = nodes_df['y'].astype('float64')
nodes_df['x'] = nodes_df['x'].astype('float64')
nodes_df = nodes_df[
    (nodes_df['y'] >= 10.74300) & (nodes_df['y'] <= 10.79326) &
    (nodes_df['x'] >= 106.67175) & (nodes_df['x'] <= 106.74505)
].copy()

# Xử lý edges
edges = edges.reset_index()
edges_df = edges[['u', 'v', 'osmid', 'highway', 'length', 'geometry']].copy()
edges_df = edges_df[
    edges_df['u'].isin(nodes_df['osmid']) &
    edges_df['v'].isin(nodes_df['osmid'])
]

def extract_first_id(val):
    val_str = str(val)
    if val_str.startswith('['):
        try:
            return int(ast.literal_eval(val_str)[0])
        except:
            pass
    try:
        return int(float(val))
    except:
        return 0
        
edges_df['osmid'] = edges_df['osmid'].apply(extract_first_id)
edges_df['u'] = edges_df['u'].astype('int64')
edges_df['v'] = edges_df['v'].astype('int64')
edges_df['osmid'] = edges_df['osmid'].astype('int64')
edges_df['length'] = edges_df['length'].astype('float64')

node_x = nodes_df.set_index('osmid')['x']
node_y = nodes_df.set_index('osmid')['y']

u_x = edges_df['u'].map(node_x)
u_y = edges_df['u'].map(node_y)
v_x = edges_df['v'].map(node_x)
v_y = edges_df['v'].map(node_y)

calc_distance = ((u_x - v_x)**2 + (u_y - v_y)**2)**0.5
edges_df.to_csv('data/edges_raw.csv', index=False)

# Xử lý file train.csv
train_df = pd.read_csv(train_file)
initial_rows = len(train_df)
train_df = train_df[['period', 'LOS', 'street_id']].copy()
train_df['period'] = train_df['period'].astype(str)
train_df['LOS'] = train_df['LOS'].astype(str)
train_df['street_id'] = train_df['street_id'].astype('int64')
los_map = {'A': 1.0, 'B': 1.24, 'C': 1.48, 'D': 1.72, 'E': 1.96, 'F': 2.2}
train_df['los_float'] = train_df['LOS'].map(los_map).fillna(1.0)
train_df.drop(columns=['LOS'], inplace=True)

unique_pairs_count = len(train_df[['street_id', 'period']].drop_duplicates()) 
duplicate_count = initial_rows - unique_pairs_count

train_df = train_df.groupby(['street_id', 'period'], as_index=False)['los_float'].mean()
train_df['los_float'] = train_df['los_float'].round(2)
def get_period_idx(p):
    parts = str(p).replace('period_', '').split('_')
    return int(parts[0]) * 2 + (1 if int(parts[1]) >= 30 else 0)
    
train_df['period_idx'] = train_df['period'].apply(get_period_idx)
train_df.to_csv('data/train.csv', index=False)


In [5]:

import math

train_pivot = train_df.pivot(index='street_id', columns='period_idx', values='los_float')

unique_osmids = set(edges_df['osmid'].unique())
unique_street_ids = set(train_pivot.index)
matched_ids = unique_osmids.intersection(unique_street_ids)

# Khởi tạo baseline 48 khung giờ với 3 đỉnh: 7.5h, 11.5h, 18h
base_los = [
    round(1.0 + 0.5*math.exp(-0.5*((i/2-7.5)/1.5)**2) + 0.25*math.exp(-0.5*((i/2-13.5)/1.5)**2) + 0.6*math.exp(-0.5*((i/2-18.0)/2.0)**2), 3) 
    for i in range(48)
]

# Tạo danh sách LOS và Weight (48 phần tử)
def build_los_list(row):
    osmid = row['osmid']
    # Phạt 0.15 cho hẻm/ngõ (nhỏ hơn 30m)
    penalty = 0.15 if row['length'] < 30 else 0.0
    
    if osmid in train_pivot.index:
        p_row = train_pivot.loc[osmid]
        return [round(p_row.get(i, base_los[i] + penalty) if pd.notna(p_row.get(i)) else base_los[i] + penalty, 3) for i in range(48)]
    else:
        return [round(base_los[i] + penalty, 3) for i in range(48)]

# Apply cho toàn bộ dòng (axis=1) để hàm lấy được cột 'length'
edges_df['los'] = edges_df.apply(build_los_list, axis=1)
edges_df['weight'] = edges_df.apply(lambda r: [round(l * r['length'], 3) for l in r['los']], axis=1)


edges_df.to_csv('data/edges.csv', index=False)

In [6]:
# Loại bỏ nodes cô đơn
connected_node_ids = set(edges_df['u']).union(set(edges_df['v']))
nodes_df = nodes_df[nodes_df['osmid'].isin(connected_node_ids)].copy()
nodes_df.to_csv('data/nodes.csv', index=False)